# Parallelism

<img src='assets/parallel_memes.png' width="500" height="650">

## Harnessing your computing power to the max

When you run a regular python program, you generally only use one core of your computer's CPU. But most modern computers have more than a single core!

<img src="assets/cpu_gpu.png" width="800" height="500"/>


Many codes and algorithms in astronomy can be categorized as "embarrassingly parallel", meaning that they can easily be split up into multiple tasks that each have little or no dependency on each other. An example of this is running the same image processing steps on multiple files, or computing the likelihood of a bunch of different sets of models. Running these tasks at the same time is "parallelsim".


Note: Parallelilization adds extra **complexity** to the code, making it harder to debug and maintain. Thus, it is important to consider the relative gain of implementing parallelization versus the extra effort in developing and maintaining that code. For this reason, we also recommend that you keep parallelization code as simple as possible. For many tasks, even the most simple parallelism is sufficient.



In [ ]:
import os
import numpy as np
import matplotlib.pylab as plt
import scipy.ndimage as ndimage
import astropy.io.fits as fits
import time
import multiprocessing as mp
%matplotlib inline

### Before we begin with multiprocessing, a few important points:

 * Only one process can spawn other processes. Your subprocesses cannot spawn their own subprocesses! Generally, there is always a way to program things so that only one process needs to spawn subprocesses
 * While python is generally OS-agnostic, you may run into OS specific issues with multiprocesses due to the fact it is implemented a bit differently for Linux, Mac, and Windows.
 * One key difference to remember when developing packages for multiple OSes: on Mac and Windows, any lines with multiprocessing calls need to be wrapped in something like ``if __name__ == "__main__"``, otherwise you will get an error mentioning something about `freeze_support()` when trying to run it. This is not required in Jupyter notebooks, so we have merely commented where you would need them here. The details for why this is necessary is related to subprocesses not spawning their own subprocesses and is quite technical, but [here is a related blog post if you want to learn more](https://pythonspeed.com/articles/python-multiprocessing/).

### Checking Resource Usage

Anytime you are testing parallelized code, it is good to monitor your CPU/RAM usage. Monitoring your CPU usage can help you assess if the number of processes being run in parallel is consistent with what you are looking for. Many times, running parallelized code already involves big data sets, and parallelization will use even more memory (you can think of it as trading runtime for memory usage). So have your resource monitor up occasionally when you are developing this code. It is also a great way to debug the code (e.g., identify hanging parallelization that is not finishing).

We create a function with an argument `index` that tells each process what chunk of the task to run. We then create a bunch of processes, give them their chunks, and call `start()` to run them. Afterwards, our main process uses `join()` to wait for each process to finish. It is important to always call `join()` at the end to ensure all processes have finished running! If a process has finished immediately, `join()` will immediately return; if a process has not finished, our main process will sleep until the process it is waiting on has finished.

### Pools

<img src="assets/multithread.jpeg" width="600" height="350"/>

Manually creating threads requires a bunch of upkeep code, which is prone to errors. In the spirit of keeping parallelism simple, use a high-level parallelization API provided by your programming language whenever possible! It will save you time and effort. For dividing up tasks with a for loop, use Python process `Pools`. Essentially, you can give
any number of tasks to a process `Pool` and the processes in the pool will loop through and do each one per your instructions.

**When in doubt about how to parallelize code, use process pools!** They are flexible and can accomodate most use cases. And since they have a standardized interface for how to use them, it will make your code more understadable by others compared to home-brewing your own parallelism.

To give "jobs" to process pools, we are using the `apply_async()` functio which we recommend as the most flexible function. An alternative is to use `map()` or one of its alternative flavors (`imap`, `starmap`), but we find this to be less flexible. Never use `apply()` if you want to parallelize your code! 

In [ ]:
mat = np.random.random((8000, 3000)) # the matrix we want to process

def matrix_pool(index):
    # divide up so that we only compute one chunk of the mat.dot(mat.T) matrix
    index_start = mat.shape[0] // 10 * index
    index_end = mat.shape[0] // 10 * (index + 1)
    val  = mat[index_start:index_end].dot(mat.T)
    print("Job {0} complete".format(index))

    return val # let's return the data this time

In [ ]:
### This entire block would need to be wrapped in `if __name__ == "__main__":`
# Synchronous parallelization

pool = mp.Pool(processes=4) # creae a pool with 4 worker processes


pool_jobs = []
for i in range(10):
    task = pool.apply_async(matrix_pool, (i,))
    pool_jobs.append(task)
    print("Created job {0}".format(i))

for i, task in enumerate(pool_jobs):
    result = task.get()
    print("Job result {0}. First value is {1}".format(i, result.ravel()[0]))

# Close the pool and wait for the work to finish
pool.close()
pool.join()

# Activity: Image Processing

Run the following code to generate some fake data and then run the following image processing function on each fake image and save the resulting data. How fast can you process the data when the images are 1000x1000? What is the speedup from 1 core to multiple cores (is it linear?)?

### Instructions

  * Run the first cell below to generate 25 images of fake data
  * Run the second cell below to run the `process_image()` function on a single image without parallelism. Time how long this takes. Let's call this time t0.
  * Write parallelization code to parallelize the `process_image()` function to process all 25 images. Time how long it takes to process 25 images. Let's call this tp.
  * Compute speedup = (25 * t0) / tp: the speedup factor between doing 25 images in series and parallelizing it.
  * (Optional) Try using different number of processes to parallelize the activity. How does the run time change with the number of processes used? How does it relate to the number of cores you have on your computer? Is it linear?

### End Product
  * Report on Piazza your best speedup factor you got. And specify how many processes you used, and how many cores you have.
  * (Optional): make a graph of wall clock time to process 25 images vs number of processes used. Post this on Piazza as well.  

### Roles
  * Driver: in charge of sharing their screen and typing the code for this activity
  * Navigator: in charge of directing the driver what to code and tracking runtime of the code (everyone else; can be more than one person)

In [ ]:
# note, this cell might take a minute to run.

def generate_fake_data(filename, dims):
    """
    Generates a fake dataframe with random numbers

    Args:
        filename (str): file to save the data to
        dims (tuple): (Ny, Nx) pair that species the size of the y and x dimensions
    """
    # some complicated random image generation. Feel free to ignore.
    # coordinate system in fourier spae
    u,v = np.meshgrid(np.fft.fftfreq(dims[1]), np.fft.fftfreq(dims[0]))
    phases = np.random.uniform(0, 2*np.pi, u.shape)
    rho = np.sqrt((u*dims[1])**2 + (v*dims[0])**2)
    # suppress high frequency by a squared exponential
    spectrum = np.exp(-rho**2/(np.max(rho)/50)**2)  * np.exp(1j * phases)
    filtered = np.real(np.fft.ifft2(spectrum))

    fits.writeto(filename, filtered, overwrite=True)


# generate fake data (can choose to save it somethere else if you want)
output_folder = os.path.join(".", "fake_data")
if not os.path.exists(output_folder):
    os.mkdir(output_folder)
fileformat = os.path.join(output_folder, "fake_{0}x{1}_{2}.fits")

ny = 1000
nx = 1000
for i in range(25):
    filename = fileformat.format(ny, nx, i)
    generate_fake_data(filename, (ny, nx) )


In [ ]:
def process_image(frame, filtersize=50):
    """
    Run a high-pass filter on the data.
    Remove the low spatial frequency (i.e., smooth features) in the image

    Args:
        frame (np.array): a 2-D image to be processed
        fitersize (int): the size of the filter. Features smaller than the filtersize will be preserved

    Returns:
        processed_frame (np.array): a 2-D image after processing
    """
    # run a median filter to smooth the image
    frame_smooth = ndimage.median_filter(frame, filtersize)

    processed_frame = frame - frame_smooth

    return processed_frame

start = time.time()

# an example of running this on one image
with fits.open(fileformat.format(ny, nx, 0)) as hdulist:
    data = hdulist[0].data


    filt_data = process_image(data)

    fig = plt.figure(figsize=(6,3))
    ax1 = fig.add_subplot(121)
    ax1.imshow(data, cmap="inferno")
    ax1.set_title("Original")
    ax2 = fig.add_subplot(122)
    ax2.imshow(filt_data, cmap="inferno")
    ax2.set_title("Filtered")

single_time = time.time() - start
print('1 image/1 process: {:.2f} seconds'.format(single_time))

#### Activity:
# write and time some code that runs this on all 25 images in parallel. How does the performance increase
# as you increase the number of processes you use?
# we recommend you use multiprocessing pool for this task

## Parallel Debugging

The recommended debugging of parallelized code is to get it working as non-parallelized code, and then wrap it in parallelism. If you have to be debugging really complicated parallelism software that you yourself wrote, ask yourself if you should really be doing this (the answer is usually no).

# Parallelism Extended Cut

There is a lot of talk about in parallelism - we are just scratching the surface. The above concepts _should_ cover the majority of parallelism use cases. However, here are some good things to know.



## Do you really need parallelism

We already emphasized this above, but parallelism adds complexity. Try avoiding parallelization until it is necessary. If some code takes 10 minutes to run, but you only need to run it once per week, is it worth parallelizing? The programming and upkeep costs may not be worth it. Parallelism also makes it hard to debug: we cannot attach the python debugger on other Python processes meaning they often crash with no error message as to why. GNU Parallel (below) is an alternative to writing more complicated code.

## GNU Parallel

If you have a script that you want to run multiple times (e.g., on multiple files), you don't need to write Python parallelization. We can use the `parallel` package that can be installed on UNIX systems to handle the parallelization. For example, if we can process a single file by running on the command line:

    python process.py file_01.fits

Then we can run it on `file_01.fits` to `file_99.fits` by running on the command line:

    parallel python process.py ::: file_{01..99}.fits


## Threads vs Processes
Python has two parallelization modules: `multiprocessing` and `multithreading`. What's the difference? Why do we only use `multiprocessing`?

Threads and processes are both tasks that your computer CPUs run in parallel. Threads share the same memory, whereas processes do not. Processes have to communicate between each other using shared variables (see below) or via a slower I/O method (writing to files, communicating over websockets). Most languages use threads to parallelize computations because sharing the same memory is very convenient and saves on resources. However, Python has the "Global Intepreter Lock" (GIL) that allows only one thread to run at a time (for complicated consistency reasons). Generally, parallelism in astronomy involves computationally intensive tasks so only one of them being able to run at a time would defeat the purpose. That's why you will generally only see multiprocessing in astronomy-related software development.



Threads are more frequently seen outside of astronomy, but it is generally useful for "I/O bound tasks" as opposed to the "CPU bound tasks" that we usually encounter in astronomy. Anytime you have tasks with a lot of sleeping/waiting time such as if you have multiple web API queries (e.g., multiple database queries) or waiting for files to be created, threads are a better choice because they use less memory and they can access all of your variables instead of having to define shared variables.

The one notable exception is that `numpy` functions that call C code releases the GIL. This means if your code is dominated by `numpy` matrix operations such as dot product, then you can actually use threads with minimal increase in runtime.

| Feature             | **Threads**                                      | **Processes**                                    |
| ---------------------------- | ------------------------------------------------ | ------------------------------------------------ |
| **Definition**               | Lightweight sub-units of a process               | Independent programs with their own memory space |
| **Memory Space**             | Shared within the same process                   | Separate for each process                        |
| **Communication**            | Fast, via shared memory                          | Slower, via Inter-Process Communication (IPC)    |
| **Isolation**                | Low (shared resources can lead to conflicts)     | High (crashes are contained to one process)      |
| **Creation Overhead**        | Low                                              | High                                             |
| **Context Switching**        | Faster (lightweight)                             | Slower (heavyweight)                             |
| **Concurrency Type**        | Better for I/O-bound tasks (in python)            | Better for CPU-bound tasks (in python)                      |
| **Crash Impact**             | Can crash the entire process                     | Usually isolated to the crashing process         |
| **Security**                 | Less secure (shared memory risks)                | More secure (separate memory spaces)             |
| **Scalability**              | Limited by GIL in CPython                        | Can fully utilize multi-core CPUs                |
| **Startup Time**             | Faster                                           | Slower                                           |
| **Communication Complexity** | Simple (direct access to shared data)            | Complex (e.g., pipes, sockets, queues)           |


## Shared Variables

Processes do not by default share memory. For processes to read/write/access the same variables, we have to declare shared memory. Python `multiprocessing` provides nearly all shared memory structure that you should be using: queues and arrays. Queues allow for interprocess communication. Arrays allow for large datasets to be accessed by multiple processes without needing to duplicate them (possibly saving a lot of memory usage). Use these special arrays and queues becasue they are automatically synchronized between processes and do not require synchronization code, which is generally low-level synchronization that we should avoid programming unless we absolutely need it.

In most applications, you will find you won't need to do this at all because each task will operate on a separte chunk of data. If you think you do, ask yourself can you rewrite it in a way that does not require passing around large chunks of data (for example, each task can open and access the data needed for its task separately). 
